# Part 1: RAG 

# Setup and Imports

In [19]:
# Regular expressions for text cleaning and sentence splitting
import re

# Hashing is used to create stable unique IDs for chunks
import hashlib

# Pandas is used to load and clean the dataset
import pandas as pd

# tqdm gives progress bars for loops
from tqdm import tqdm

# ChromaDB stores and searches the embedded medical chunks
import chromadb

# SentenceTransformer converts text into numerical embeddings
from sentence_transformers import SentenceTransformer

# Load Dataset

In [20]:
# Load the MedQuAD dataset from the data folder.
# Make sure your CSV file is located at data/medquad.csv.
df = pd.read_csv("medquad.csv")

# Standardize column names:
# lowercase, remove extra spaces, and replace spaces with underscores.
df.columns = [c.lower().strip().replace(" ", "_") for c in df.columns]

# Print column names so we can confirm the dataset has question and answer columns.
print(df.columns)

# Display the first five rows to inspect the data.
df.head()

Index(['question', 'answer', 'source', 'focus_area'], dtype='object')


,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma


# Preprocessing

In [21]:
def clean_text(text):
    """
    Clean a text field by:
    - handling missing values
    - converting the input to a string
    - removing extra whitespace
    """
    if pd.isna(text):
        return ""
    
    return re.sub(r"\s+", " ", str(text)).strip()


# Clean the question and answer columns.
df["question"] = df["question"].apply(clean_text)
df["answer"] = df["answer"].apply(clean_text)

# Remove rows where either the question or answer is empty.
df = df[(df["question"] != "") & (df["answer"] != "")]

# Remove duplicate question-answer pairs.
df = df.drop_duplicates(subset=["question", "answer"])

# Print the final number of usable QA pairs.
print("Cleaned dataset size:", len(df))

Cleaned dataset size: 16359


# Chunking Strategy

In [22]:
def sentence_split(text):
    """
    Split text into sentences using punctuation marks.
    This helps keep chunks readable instead of cutting text randomly.
    """
    return re.split(r'(?<=[.!?])\s+', text)


def chunk_text(text, max_words=220, overlap_words=40):
    """
    Split a long medical answer into smaller overlapping chunks.

    max_words:
        Maximum approximate number of words per chunk.

    overlap_words:
        Number of words repeated from the previous chunk.
        This helps preserve context across chunks.
    """
    sentences = sentence_split(text)

    # Stores all final chunks
    chunks = []

    # Temporarily stores sentences for the current chunk
    current = []

    for sent in sentences:
        # Count words already in the current chunk
        current_words = " ".join(current).split()

        # Count words in the next sentence
        sent_words = sent.split()

        # If adding this sentence keeps the chunk under the word limit,
        # add it to the current chunk.
        if len(current_words) + len(sent_words) <= max_words:
            current.append(sent)

        # Otherwise, save the current chunk and start a new one.
        else:
            if current:
                chunks.append(" ".join(current))

            # Keep some words from the previous chunk for overlap/context.
            overlap = " ".join(current_words[-overlap_words:])

            # Start the new chunk with overlap + the new sentence.
            current = [overlap, sent] if overlap else [sent]

    # Add the final chunk if anything remains.
    if current:
        chunks.append(" ".join(current))

    # Remove empty chunks and extra whitespace.
    return [c.strip() for c in chunks if c.strip()]

# Test Chunking

In [23]:
# Select the first answer in the dataset as an example.
example = df.iloc[0]["answer"]

# Chunk the example answer.
chunks = chunk_text(example)

# Print how many chunks were created.
print("Number of chunks:", len(chunks))

# Print the first chunk to inspect quality.
print(chunks[0])

Number of chunks: 2
Glaucoma is a group of diseases that can damage the eye's optic nerve and result in vision loss and blindness. While glaucoma can strike anyone, the risk is much greater for people over 60. How Glaucoma Develops There are several different types of glaucoma. Most of these involve the drainage system within the eye. At the front of the eye there is a small space called the anterior chamber. A clear fluid flows through this chamber and bathes and nourishes the nearby tissues. (Watch the video to learn more about glaucoma. To enlarge the video, click the brackets in the lower right-hand corner. To reduce the video, press the Escape (Esc) button on your keyboard.) In glaucoma, for still unknown reasons, the fluid drains too slowly out of the eye. As the fluid builds up, the pressure inside the eye rises. Unless this pressure is controlled, it may cause damage to the optic nerve and other parts of the eye and result in loss of vision. Open-angle Glaucoma The most common 

# Build Vector Database

In [24]:
# Create an in-memory ChromaDB client.
# This stores the vector database while the notebook is running.
client = chromadb.PersistentClient(path="chroma_db")

# Create or load a collection named medical_qa.
# A collection is like a table of embedded documents.
collection = client.get_or_create_collection(
    name="medical_qa",
    metadata={"hnsw:space": "cosine"}
)

# Load the embedding model.
# This model converts text into vectors that can be searched semantically.
model = SentenceTransformer("all-MiniLM-L6-v2")

/opt/anaconda3/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


# Convert Dataset Into Chunk Records

In [25]:
def make_id(text):
    """
    Create a stable unique ID from text.
    This helps each original QA pair and chunk have a consistent identifier.
    """
    return hashlib.md5(text.encode()).hexdigest()


# This list will store all chunk-level records before inserting into ChromaDB.
records = []

# Loop through every row in the cleaned dataset.
for i, row in tqdm(df.iterrows(), total=len(df)):

    # Get the original medical question and answer.
    question = row["question"]
    answer = row["answer"]

    # Create one source ID for the full question-answer pair.
    source_id = make_id(question + answer)

    # Split the answer into smaller chunks.
    chunks = chunk_text(answer)

    # Create a separate record for each chunk.
    for j, chunk in enumerate(chunks):

        # Each chunk gets a unique ID.
        chunk_id = f"{source_id}_{j}"

        records.append({
            # Unique chunk ID
            "id": chunk_id,

            # Text that will be embedded and searched
            "document": f"Question: {question}\nAnswer: {chunk}",

            # Metadata is extra information stored with the chunk.
            # This is useful for citations and filtering later.
            "metadata": {
                "question": question,
                "source": str(row.get("source", "")),
                "question_type": str(row.get("question_type", ""))
            }
        })

# Show how many chunks were created.
print("Total chunk records:", len(records))

100%|██████████| 16359/16359 [00:00<00:00, 18464.92it/s]

Total chunk records: 25669


# Add Chunks to ChromaDB

In [26]:
# Number of chunks processed at a time.
# Batching prevents memory issues.
batch_size = 64

# Loop through the records in batches.
for i in tqdm(range(0, len(records), batch_size)):

    # Select one batch of records.
    batch = records[i:i+batch_size]

    # Extract text, IDs, and metadata from the batch.
    docs = [r["document"] for r in batch]
    ids = [r["id"] for r in batch]
    metas = [r["metadata"] for r in batch]

    # Convert each document into an embedding vector.
    embeddings = model.encode(docs).tolist()

    # Insert or update the batch in ChromaDB.
    collection.upsert(
        documents=docs,
        ids=ids,
        metadatas=metas,
        embeddings=embeddings
    )

print("Vector DB built!")

100%|██████████| 402/402 [01:12<00:00,  5.54it/s]

Vector DB built!


# Retrieval Function

In [27]:
def search_medical_kb(query, top_k=5):
    """
    Search the medical knowledge base for the most relevant chunks.

    query:
        The user's medical question.

    top_k:
        Number of retrieved chunks to return.

    returns:
        A list of dictionaries containing retrieved text, metadata, and score.
    """

    # Convert the user query into an embedding vector.
    query_embedding = model.encode([query]).tolist()[0]

    # Search ChromaDB for the most similar chunks.
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    # Store formatted retrieval results.
    output = []

    # Loop through retrieved results.
    for i in range(len(results["ids"][0])):

        # Convert distance into a simple similarity-like score.
        # Smaller distance means better match.
        score = 1 / (1 + results["distances"][0][i])

        output.append({
            "text": results["documents"][0][i],
            "question": results["metadatas"][0][i]["question"],
            "source": results["metadatas"][0][i]["source"],
            "score": score
        })

    return output

# Test Retrieval

In [28]:
def search_medical_kb(query, top_k=5):
    query_embedding = model.encode([query]).tolist()[0]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    output = []

    for i in range(len(results["ids"][0])):
        output.append({
            "text": results["documents"][0][i],
            "question": results["metadatas"][0][i]["question"],
            "source": results["metadatas"][0][i]["source"],
            "score": float(1 / (1 + results["distances"][0][i]))
        })

    return output

# Validate Retrieval Quality 

In [29]:
queries = [
    "What are symptoms of diabetes?",
    "How is asthma treated?",
    "What causes high blood pressure?",
    "What are side effects of ibuprofen?",
    "How do you prevent heart disease?"
]

for q in queries:
    print("\n\nQUERY:", q)
    results = search_medical_kb(q, top_k=3)

    for r in results:
        print("- Score:", round(r["score"], 3))
        print("  Question:", r["question"])
        print("  Text:", r["text"][:200])



QUERY: What are symptoms of diabetes?
- Score: 0.838
  Question: What are the symptoms of Diabetes ?
  Text: Question: What are the symptoms of Diabetes ?
Answer: Many people with diabetes experience one or more symptoms, including extreme thirst or hunger, a frequent need to urinate and/or fatigue. Some los
- Score: 0.829
  Question: What are the symptoms of Diabetes ?
  Text: Question: What are the symptoms of Diabetes ?
Answer: Diabetes is often called a "silent" disease because it can cause serious complications even before you have symptoms. Symptoms can also be so mild
- Score: 0.81
  Question: What are the symptoms of Your Guide to Diabetes: Type 1 and Type 2 ?
  Text: Question: What are the symptoms of Your Guide to Diabetes: Type 1 and Type 2 ?
Answer: The signs and symptoms of diabetes are - being very thirsty - urinating often - feeling very hungry - feeling ver


QUERY: How is asthma treated?
- Score: 0.76
  Question: What are the treatments for Asthma ?
  Text: Question: